In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pickle

# set the font size
plt.rcParams.update({'font.size': 7})
# set Helvetica globally
plt.rcParams['font.family'] = 'Helvetica'

plot_folder = "../plots/18_calculate_SPS"
os.makedirs(plot_folder, exist_ok=True)

from nupack import *
my_model = Model(material='rna', celsius=37)

In [ ]:
mirbase = pd.read_csv("../microrna_data/mirbase_extended.csv", index_col=0)

In [ ]:
# get likely real mirnas
with open("../microrna_data/likely_real_mirnas.pkl", "rb") as f:
    likely_real_mirnas = pickle.load(f)

In [ ]:
mirbase = mirbase[mirbase.index.isin(likely_real_mirnas)]
mirbase["seed_seq"] = mirbase["sequence_orig"].str[1:8]
mirbase["seed_seq"] = mirbase["seed_seq"].str.replace("T", "U")
mirbase["seed_target"] = mirbase["seed_seq"].apply(lambda x: reverse_complement(x, "RNA"))
unique_seeds = mirbase["seed_seq"].unique()

In [ ]:
# to get the regular ddG comparable to other calculations, RT ln(ρH2O/1 M) needs to be added to values
NUPACK_ENERGY_SHIFT = 2.47

seed_ddGs = {}
for seed_seq in unique_seeds:
    seed_target = reverse_complement(seed_seq, "RNA")
    
    # Define strand species
    s_mir = Strand(seed_seq, name='mir')
    s_tar = Strand(seed_target, name='seq')

    set1 = ComplexSet(strands=[s_mir, s_tar],
                complexes=SetSpec(max_size=2, exclude=[[s_mir, s_mir], [s_tar, s_tar]]))

    complex_results = complex_analysis(complexes=set1, model=my_model, compute=['pfunc'])
    
    dG_mir = complex_results["(mir)"].free_energy
    dG_seq = complex_results["(seq)"].free_energy
    if "(mir+seq)" in str(complex_results.keys()):
        dG_complex = complex_results["(mir+seq)"].free_energy
    else:
        dG_complex = complex_results["(seq+mir)"].free_energy

    seed_ddGs[seed_seq] = dG_complex + NUPACK_ENERGY_SHIFT

In [ ]:
minimum = max(seed_ddGs.values())
for key, val in seed_ddGs.items():
    if val == minimum:
        print(key)
        break

In [ ]:
plt.figure(figsize=(2.0,1.8))
plt.hist(seed_ddGs.values(), bins = np.arange(-20, 0, 1), edgecolor='black')
plt.xlabel(r"Predicted SPS (kcal/mole)")
plt.ylabel("microRNA count")
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, "SPS_distribution.svg"))

In [ ]:
mirbase["SPS"] = mirbase["seed_seq"].map(seed_ddGs)
mirbase.to_csv("../microrna_data/mirbase_extended_with_SPS.csv")

# Compare with deviation

In [ ]:
mirbase = pd.read_csv("../microrna_data/mirbase_extended_with_SPS.csv", index_col=0)

In [ ]:
cell_lines_subset = ["HEK293T", "HeLa", "SKNSH", "MCF7", "HUH7", "A549"]
cell_lines_rest = ["HaCaT", "PC3"] # leave out "JEG3", "Tera1"
cell_lines_measured = cell_lines_subset + cell_lines_rest

In [ ]:
# load popt
with open("../outputs/3_fitting/combined_dataset/combined_dataset_popt.pkl", "rb") as f:
    popt = pickle.load(f)

from library2_utils.transfer_functions import transfer_function

In [ ]:
knockdown_filter = pd.read_csv("../outputs/3_fitting/combined_dataset_filter/combined_dataset_filter_knockdown.csv", index_col=0)

In [ ]:
deviation_df = pd.read_csv("../outputs/3_fitting/combined_dataset/combined_dataset_deviation_bias_aware.csv", index_col=0)
deviation_df = deviation_df.loc[knockdown_filter.index]

In [ ]:
# expression filtering
df_merge_crosstalk_filter = pd.read_csv("../microrna_data/3_output/Alles_Keller_combined_expression_congruent.csv", index_col=0).dropna()

# remove "hsa-miR-3613-3p" if present
if "hsa-miR-3613-3p" in df_merge_crosstalk_filter.index:
    df_merge = df_merge_crosstalk_filter.drop("hsa-miR-3613-3p", axis=0)

Deviation is calculated as measured - predicted. Thus, a positive value implies less knockdown than expected.

In [ ]:
%%capture output
all_x_vals = {}
all_y_vals = {}
outliers_by_cell_line = {}
mirnas_by_cell_line = {}

for cell_line in cell_lines_measured:
    expression_filter = df_merge_crosstalk_filter[df_merge_crosstalk_filter[cell_line] > 3.5].index
    deviation_df_filtered = deviation_df[deviation_df.index.isin(expression_filter)]
    mirnas_by_cell_line[cell_line] = deviation_df_filtered.index
    x_vals = mirbase.loc[deviation_df_filtered.index, "SPS"].values
    y_vals = deviation_df_filtered[cell_line].values
    outliers_by_cell_line[cell_line] = mirbase.loc[deviation_df_filtered.index, "SPS"] > -6
    outliers_by_cell_line[cell_line] = outliers_by_cell_line[cell_line][outliers_by_cell_line[cell_line]].index

    all_x_vals[cell_line] = mirbase.loc[deviation_df_filtered.index, "SPS"]
    all_y_vals[cell_line] = deviation_df_filtered[cell_line]

    plt.figure(figsize=(3, 2.4))
    plt.scatter(x_vals, y_vals, s=2)
    r2 = np.corrcoef(x_vals, y_vals)[0, 1]
    plt.title(f"{cell_line}, $r$: {r2:.2f}\nrestricted to expression > " + r"$10^{3.5}$")
    plt.xlabel(r"Predicted SPS (kcal/mole)")
    plt.ylabel("Deviation")
    plt.tight_layout()
    plt.savefig(os.path.join(plot_folder, f"18_SPS_vs_deviation_{cell_line}.png"), dpi=300)

In [ ]:
all_outliers = []
for cell_line in cell_lines_measured:
    print(cell_line, len(outliers_by_cell_line[cell_line]), "outliers:", outliers_by_cell_line[cell_line].tolist())
    all_outliers.extend([e for e in outliers_by_cell_line[cell_line].tolist() if 'hsa-miR-30' in e or 'hsa-miR-374' in e])

In [ ]:
all_filtered_mirnas = [val for mirna_list in mirnas_by_cell_line.values() for val in mirna_list]
mirbase_filter = mirbase.loc[all_filtered_mirnas]

In [ ]:
all_x_vals = pd.DataFrame(all_x_vals)
all_y_vals = pd.DataFrame(all_y_vals)

In [ ]:
all_x_vals = all_x_vals.unstack()
all_y_vals = all_y_vals.unstack()

In [ ]:
plt.figure(figsize=(3, 2.4))
plt.scatter(all_x_vals, all_y_vals, s=2)
r = np.corrcoef(all_x_vals, all_y_vals)[0, 1]
plt.title(f"$r$: {r:.2f}\nrestricted to expression > " + r"$10^{3.5}$")
plt.xlabel(r"Predicted SPS (kcal/mole)")
plt.ylabel("Deviation from fit")
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"19_SPS_vs_deviation_all.png"), dpi=300)

In [ ]:
# create a fit
# Fit a line to all data points
x_vals_line = np.arange(-13,-2,1)
fit_all = np.polyfit(all_x_vals.dropna(), all_y_vals.dropna(), 1)
fit_line_all = np.poly1d(fit_all)
all_y_fit = fit_line_all(x_vals_line)

all_x_vals_filter = all_x_vals[~all_x_vals.index.get_level_values(1).isin(all_outliers)].dropna()
all_y_vals_filter = all_y_vals[~all_y_vals.index.get_level_values(1).isin(all_outliers)].dropna()

all_xvals_other = all_x_vals[all_x_vals.index.get_level_values(1).isin(all_outliers)].dropna()
all_yvals_other = all_y_vals[all_y_vals.index.get_level_values(1).isin(all_outliers)].dropna()

# Fit a line to filtered data points
fit_filter = np.polyfit(all_x_vals_filter, all_y_vals_filter, 1)
fit_line_filter = np.poly1d(fit_filter)
filtered_y_fit = fit_line_filter(x_vals_line)

plt.figure(figsize=(3, 2))
plt.scatter(all_x_vals_filter, all_y_vals_filter, s=2, color="tab:blue", rasterized=True)
plt.scatter(all_xvals_other, all_yvals_other, s=2, color="tab:red", label="miR-30 or miR-374 family", rasterized=True)

r_all = np.corrcoef(all_x_vals.dropna(), all_y_vals.dropna())[0, 1]
r_filter = np.corrcoef(all_x_vals_filter, all_y_vals_filter)[0, 1]

# Plot linear fits
plt.plot(x_vals_line, all_y_fit, color="tab:red", linestyle="--", label=f"Fit (All Data), r={r_all:.2f}")
plt.plot(x_vals_line, filtered_y_fit, color="tab:blue", linestyle="--", label=f"Fit (Filtered Data), r={r_filter:.2f}")

plt.title(f"restricted to miRNA expression > " + r"$10^{3.5}$ rpm")
plt.xlabel(r"Predicted SPS (kcal/mole)")
plt.ylabel("Deviation from fit")
plt.tight_layout()
plt.legend(loc=[1,0.5])
plt.savefig(os.path.join(plot_folder, f"19_SPS_vs_deviation_all_filter.svg"), dpi=400)